# Sprint 12 - Cheap Levers (TTA + Class-Balanced Sampling)

**Goal:** close the last ~0.03-0.035 of field F1 (0.565 -> 0.60) on repeat=20 with TWO cheap,
already-queued levers. Web research showed the 74-92% field SOTA is ARCHITECTURAL (ViT/Mamba/
ConvNeXt + GAN augmentation + curated data) and out of scope for this semester. So this sprint
does an HOUR of cheap levers max, then ships repeat=20 as-is and moves to the frontend + paper.

**Bar to beat (Sprint 10):** `best_plantvillage_s10_blr20.pt`
  - Lab accuracy 0.989 / Field accuracy 0.635 (macro F1 field 0.565)

**Lever 1 - TTA (test-time augmentation):** already in `ml/evaluate.py --tta` (avg softmax over
image + horizontal flip). No code change; just re-eval with the flag as new `s12_tta_*` rows.
Keep OFF for apples-to-apples ablation unless the whole table is re-run consistently.

**Lever 2 - class-balanced sampling:** NEW code in `ml/train.py --class-balanced` (and
`ml/data_loading.py`). Replaces uniform shuffle with a WeightedRandomSampler using 1/class-frequency
weights, calibrated PER constituent (PlantVillage and PlantDoc each weighted against their own
distribution) so minority classes in the imbalanced PlantDoc slice get a fair share every epoch.

**Stop point:** if field F1 does not clear 0.60 here, ship repeat=20 as-is. Do NOT chase 74-92%
field SOTA on this semester's clock (parked in plan.md as future work).

**Plan:**
  1. Setup + hydrate + split + replay label fixes (same as Sprint 10/11)
  2. Lever 1: TTA eval of repeat=20 on field + lab (new `s12_tta_*` rows)
  3. Lever 2: class-balanced re-train of the repeat=20 recipe (one run)
  4. Eval the class-balanced model on field + lab (`s12_cb_*` rows)
  5. Compare all vs repeat=20 baseline; pick the best; if nothing clears 0.60, ship repeat=20.


In [ ]:
import platform, subprocess, sys
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)


## Step 1 - Mount Drive + clone repo + install deps

Same as Sprint 11: mount Drive (checkpoints + results live there), clone/pull the repo, define
paths and the `run` helper.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")
LOCAL_RAW_DIR = Path("/content/folium_raw")
LOCAL_DATA_DIR = Path("/content/folium_data")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

def run(cmd, cwd, label, stream=False):
    import os, subprocess
    env = dict(os.environ, PYTHONPATH=str(REPO_DIR))
    if stream:
        proc = subprocess.Popen(cmd, cwd=str(cwd), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        lines = []
        for line in proc.stdout:
            print(line, end='')
            lines.append(line)
        proc.wait()
        combined = ''.join(lines)
        if proc.returncode != 0:
            print(f"\n[{label}] failed (returncode {proc.returncode})")
        assert proc.returncode == 0, label
        class _Result:
            def __init__(self, code, out): self.returncode, self.stdout = code, out
        return _Result(proc.returncode, combined)
    proc = subprocess.run(cmd, cwd=str(cwd), env=env, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"[{label}] failed (returncode {proc.returncode})")
        print(proc.stdout[-2000:])
        print(proc.stderr[-2000:])
        assert proc.returncode == 0, label
    return proc


In [ ]:
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn


## Step 2 - Clean old `s12_` rows from ablation CSV

Remove previous `s12_` rows so re-runs do not pile up duplicate variants with the same names.


In [ ]:
import pandas as pd

csv_path = RESULTS_DIR / "ablation_results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    drop = df["variant"].str.startswith("s12_")
    n = int(drop.sum())
    if n > 0:
        df = df[~drop].reset_index(drop=True)
        df.to_csv(csv_path, index=False)
        print(f"Removed {n} old s12_ rows")
    else:
        print("No old s12_ rows found.")
else:
    print("No ablation CSV yet.")


## Step 3 - Hydrate raw from Drive + organize splits + replay label fixes

Same as Sprint 10/11. Replaying the 7 verified label fixes keeps PlantDoc test labels correct.


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--seed", "42",
    "--val-fraction", "0.15",
    "--test-fraction", "0.15",
], cwd=str(REPO_DIR), label="organize_datasets.py failed")
print("Data ready at", LOCAL_DATA_DIR)


In [ ]:
import pandas as pd
import shutil

fixes_csv = RESULTS_DIR / "s10_label_fixes.csv"
test_dir = LOCAL_DATA_DIR / "plantdoc" / "test"

if fixes_csv.exists():
    fixes = pd.read_csv(fixes_csv)
    moved = done = missing = 0
    for _, row in fixes.iterrows():
        src = test_dir / str(row["src_class"]) / str(row["filename"])
        dst = test_dir / str(row["dst_class"]) / str(row["filename"])
        if dst.exists():
            done += 1
        elif src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            moved += 1
        else:
            missing += 1
    print(f"label fixes: {moved} moved, {done} already in place, {missing} missing")
else:
    print("No label fixes CSV; skipping.")


## Step 4 - Lever 1: TTA eval of the repeat=20 baseline

Re-run `ml.evaluate` on the shipped checkpoint with `--tta` (horizontal-flip softmax averaging)
on BOTH field and lab. Logs `s12_tta_field` / `s12_tta_lab` rows. No code change - the flag
already exists.


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

BASELINE = CHECKPOINT_DIR / "best_plantvillage_s10_blr20.pt"

def eval_row(ckpt_path, dataset, split, variant, extra=()):
    args = [
        "--checkpoint", str(ckpt_path),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", split,
        "--variant", variant,
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ] + list(extra)
    cmd = [sys.executable, "-m", "ml.evaluate"] + args
    return run(cmd, cwd=str(REPO_DIR), label=f"eval {variant} failed", stream=True)

print("4a) TTA field (PlantDoc)")
eval_row(BASELINE, "plantdoc", "test", "s12_tta_field", ["--map-to-pv", "--tta"])
print("4b) TTA lab (PlantVillage)")
eval_row(BASELINE, "plantvillage", "test", "s12_tta_lab", ["--tta"])
print("Done TTA eval.")


## Step 5 - Lever 2: class-balanced re-train of the repeat=20 recipe

Same recipe as Sprint 10 repeat=20 (resnet50, mix-with plantdoc, repeat=20, backbone-lr 1e-5,
unfreeze 10, augment) but adds ONE new flag: `--class-balanced`. If GPU quota is gone, skip
this step and just ship repeat=20 (TTA may already clear the bar). New tag `s12_cb`.

NOTE: re-running from scratch takes GPU time; if quota is exhausted this step is the bottleneck.


In [ ]:
import sys
print("=" * 60)
print("5) backbone-lr=1e-5, mix-with=plantdoc, repeat=20, class-balanced")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--lr", "1e-3",
    "--backbone-lr", "1e-5",
    "--unfreeze-blocks", "10",
    "--batch-size", "128",
    "--augment",
    "--mix-with", "plantdoc",
    "--plantdoc-repeat", "20",
    "--class-balanced",
    "--tag", "s12_cb",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="class-balanced train failed", stream=True)


## Step 6 - Evaluate the class-balanced model on field + lab

Only if Step 5 produced a `best_plantvillage_s12_cb.pt`. Logs `s12_cb_field` / `s12_cb_lab`.


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

CB = CHECKPOINT_DIR / "best_plantvillage_s12_cb.pt"

def eval_row(ckpt_path, dataset, split, variant, extra=()):
    args = [
        "--checkpoint", str(ckpt_path),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", split,
        "--variant", variant,
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ] + list(extra)
    cmd = [sys.executable, "-m", "ml.evaluate"] + args
    return run(cmd, cwd=str(REPO_DIR), label=f"eval {variant} failed", stream=True)

if CB.exists():
    print("6a) class-balanced field (PlantDoc)")
    eval_row(CB, "plantdoc", "test", "s12_cb_field", ["--map-to-pv"])
    print("6b) class-balanced lab (PlantVillage)")
    eval_row(CB, "plantvillage", "test", "s12_cb_lab", [])
else:
    print("No best_plantvillage_s12_cb.pt - Step 5 skipped (GPU quota?) or failed. Nothing to eval.")


## Step 7 - Compare TTA + class-balanced vs repeat=20 baseline

Pull the `s12_` rows and the Sprint 10 `s10_blr20_best_*` rows. Decide on FIELD F1 (primary
goal is clearing 0.60) while keeping lab high. If nothing clears 0.60, ship repeat=20 as-is.


In [ ]:
import pandas as pd
csv_path = RESULTS_DIR / "ablation_results.csv"
df = pd.read_csv(csv_path)

def row(variant):
    r = df[df["variant"] == variant]
    return r.iloc[0] if len(r) else None

def pretty(variant, name):
    r = row(variant)
    if r is None:
        print(f"  {name:<26} MISSING ({variant})")
        return None
    print(f"  {name:<26} acc={r['accuracy']:.4f}  prec={r['precision']:.4f}  recall={r['recall']:.4f}  f1={r['f1']:.4f}")
    return r

print("=== BASELINE: Sprint 10 repeat=20 ===")
f20 = pretty("s10_blr20_best_field", "repeat=20 field")
l20 = pretty("s10_blr20_best_lab", "repeat=20 lab")

print("\n=== LEVER 1: TTA ===")
ft = pretty("s12_tta_field", "TTA field")
lt = pretty("s12_tta_lab", "TTA lab")

print("\n=== LEVER 2: class-balanced ===")
fc = pretty("s12_cb_field", "class-balanced field")
lc = pretty("s12_cb_lab", "class-balanced lab")

def check(acc, f1, name):
    if acc is None:
        print(f"\n  {name}: no result")
        return
    goal = "CLEARS 0.60 + lab held" if f1 >= 0.60 else "below 0.60 field F1"
    print(f"  {name} field F1 = {f1:.4f} -> {goal}")

print("\n=== VERDICT ===")
if ft is not None:
    check(ft["accuracy"], ft["f1"], "TTA")
if fc is not None:
    check(fc["accuracy"], fc["f1"], "class-balanced")
print("\nIf neither clears 0.60, ship repeat=20 as-is and move to the frontend/paper.")
